In [1]:
# ============================================================
# CELL 1 — Install dependencies
# Run this cell first, then restart runtime when prompted
# ============================================================

!pip install -q chromadb rank_bm25 sentence-transformers openai scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

In [2]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/invalid_detect/data/chroma_db', exist_ok=True)
os.makedirs('/content/drive/MyDrive/invalid_detect/results', exist_ok=True)

DRIVE_DIR   = '/content/drive/MyDrive/invalid_detect'
CHROMA_DIR  = f'{DRIVE_DIR}/data/chroma_db'
RESULTS_DIR = f'{DRIVE_DIR}/results'
print(f"Drive mounted → {DRIVE_DIR}")

Mounted at /content/drive
Drive mounted → /content/drive/MyDrive/invalid_detect


In [3]:
# ============================================================
# CELL 3 — Upload your files
# Only needed once — files stay in Drive after first upload
# ============================================================

from google.colab import files
import shutil

print("Upload your 5 XML files + full-edkII-dd.csv")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(f'/content/{fname}', f'{DRIVE_DIR}/data/{fname}')
    print(f"  Saved: {DRIVE_DIR}/data/{fname}")

Upload your 5 XML files + full-edkII-dd.csv


Saving full-edkII-dd.csv to full-edkII-dd.csv
Saving tianocore_closed_0_1000.xml to tianocore_closed_0_1000.xml
Saving tianocore_closed_1000_2000.xml to tianocore_closed_1000_2000.xml
Saving tianocore_closed_2000_3000.xml to tianocore_closed_2000_3000.xml
Saving tianocore_closed_3000_4000.xml to tianocore_closed_3000_4000.xml
Saving tianocore_closed_4000_5000.xml to tianocore_closed_4000_5000.xml
  Saved: /content/drive/MyDrive/invalid_detect/data/full-edkII-dd.csv
  Saved: /content/drive/MyDrive/invalid_detect/data/tianocore_closed_0_1000.xml
  Saved: /content/drive/MyDrive/invalid_detect/data/tianocore_closed_1000_2000.xml
  Saved: /content/drive/MyDrive/invalid_detect/data/tianocore_closed_2000_3000.xml
  Saved: /content/drive/MyDrive/invalid_detect/data/tianocore_closed_3000_4000.xml
  Saved: /content/drive/MyDrive/invalid_detect/data/tianocore_closed_4000_5000.xml


In [4]:
# ============================================================
# CELL 4 — Config & API keys
# ============================================================

import os
from google.colab import userdata

# ── OpenAI key ────────────────────────────────────────────────
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("OpenAI API key loaded")
except:
    OPENAI_API_KEY = input("Paste your OpenAI API key: ").strip()
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# ── Anthropic key (for Claude models) ────────────────────────
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY') or ""
    print("Anthropic API key loaded")
except:
    ANTHROPIC_API_KEY = ""
    print("No Anthropic API key — Claude models will be skipped")

# ── Org + project ID for OpenAI opt-out header ───────────────
try:
    OPENAI_ORG_ID     = userdata.get('OPENAI_ORG_ID')     or ""
    OPENAI_PROJECT_ID = userdata.get('OPENAI_PROJECT_ID') or ""
    print(f"OpenAI org ID    : {'set' if OPENAI_ORG_ID     else 'not set'}")
    print(f"OpenAI project ID: {'set' if OPENAI_PROJECT_ID else 'not set'}")
except:
    OPENAI_ORG_ID     = ""
    OPENAI_PROJECT_ID = ""

# ── All models to run ─────────────────────────────────────────
OPENAI_MODELS = [
    "gpt-4o-mini",
    "gpt-4o",
    "gpt-4.1-mini",
    "gpt-4.1",
    "gpt-5.4-mini",
    "gpt-5.4",
    "gpt-5.5",
]

CLAUDE_MODELS = [
    "claude-haiku-4-5",
    "claude-sonnet-4-6",
    "claude-opus-4-7",
] if ANTHROPIC_API_KEY else []

ALL_MODELS = OPENAI_MODELS + CLAUDE_MODELS

print(f"\nModels to run ({len(ALL_MODELS)} total):")
for m in ALL_MODELS:
    provider = "Anthropic" if m.startswith("claude") else "OpenAI"
    print(f"  [{provider}] {m}")

XML_FILES = [
    f'{DRIVE_DIR}/data/tianocore_closed_0_1000.xml',
    f'{DRIVE_DIR}/data/tianocore_closed_1000_2000.xml',
    f'{DRIVE_DIR}/data/tianocore_closed_2000_3000.xml',
    f'{DRIVE_DIR}/data/tianocore_closed_3000_4000.xml',
    f'{DRIVE_DIR}/data/tianocore_closed_4000_5000.xml',
]
GITHUB_CSV    = f'{DRIVE_DIR}/data/full-edkII-dd.csv'
XML_CONTEXT_K = 3

print("\nConfig ready.")


OpenAI API key loaded
Anthropic API key loaded
OpenAI org ID    : set
OpenAI project ID: set

Models to run (10 total):
  [OpenAI] gpt-4o-mini
  [OpenAI] gpt-4o
  [OpenAI] gpt-4.1-mini
  [OpenAI] gpt-4.1
  [OpenAI] gpt-5.4-mini
  [OpenAI] gpt-5.4
  [OpenAI] gpt-5.5
  [Anthropic] claude-haiku-4-5
  [Anthropic] claude-sonnet-4-6
  [Anthropic] claude-opus-4-7

Config ready.


In [5]:
# ============================================================
# CELL 5 — Ingest: parse XML corpus + GitHub issues
# ============================================================

import re
import xml.etree.ElementTree as ET
import pandas as pd


def _clean(text):
    text = re.sub(r"https?://\S+", "<URL>", text or "")
    return re.sub(r"\s+", " ", text).strip()


def _xml_body(bug_el):
    bodies = []
    for ld in bug_el.findall("long_desc"):
        t = ld.find("thetext")
        if t is not None and t.text:
            bodies.append(t.text.strip())
        if len(bodies) >= 3:
            break
    return "\n\n".join(bodies)


def load_xml_corpus(xml_paths):
    bugs = []
    for path in xml_paths:
        tree = ET.parse(path)
        root = tree.getroot()
        for bug_el in root.findall("bug"):
            def t(tag):
                n = bug_el.find(tag)
                return (n.text or "").strip() if n is not None else ""
            title      = t("short_desc")
            body       = _xml_body(bug_el)
            resolution = t("resolution")
            bugs.append(dict(
                id         = t("bug_id"),
                title      = title,
                body       = body,
                package    = t("cf_package"),
                priority   = t("priority"),
                resolution = resolution,
                text       = _clean(f"Title: {title}\n\nDescription: {body}"),
            ))
    print(f"[ingest] XML corpus: {len(bugs):,} bugs from {len(xml_paths)} files")
    return bugs


def load_github_issues(csv_path):
    df = pd.read_csv(csv_path)
    issues = []
    for _, row in df.iterrows():
        title    = str(row.get("Title", "") or "")
        package  = str(row.get("package", "") or "")
        itype    = str(row.get("type", "") or "")
        comments = int(float(str(row.get("Comments", 0) or 0)))
        state    = str(row.get("state", "") or "")
        iid      = str(int(row["Issue Number"]))
        issues.append(dict(
            id       = iid,
            title    = title,
            package  = package,
            type     = itype,
            comments = comments,
            state    = state,
            url      = str(row.get("URL", "") or ""),
            text     = _clean(
                f"Title: {title}\nPackage: {package}\n"
                f"Comments: {comments}\nState: {state}"
            ),
        ))
    print(f"[ingest] GitHub issues: {len(issues):,} issues loaded")
    return issues


xml_bugs  = load_xml_corpus(XML_FILES)
gh_issues = load_github_issues(GITHUB_CSV)

# Stats
from collections import Counter
invalid_xml = [b for b in xml_bugs if b.get('resolution','').upper() == 'INVALID']
valid_xml   = [b for b in xml_bugs if b.get('resolution','').upper() == 'FIXED']
print(f"\nXML corpus:")
print(f"  INVALID resolution : {len(invalid_xml)}")
print(f"  FIXED resolution   : {len(valid_xml)}")
print(f"\nGitHub issues: {len(gh_issues)}")

[ingest] XML corpus: 3,529 bugs from 5 files
[ingest] GitHub issues: 2,610 issues loaded

XML corpus:
  INVALID resolution : 182
  FIXED resolution   : 2962

GitHub issues: 2610


In [6]:
# ============================================================
# CELL 6 — Build retriever (ChromaDB + BM25)
# XML corpus only — used to retrieve similar historical bugs
# Run once — index persists to Drive
# ============================================================

import torch
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm

EMBED_MODEL    = "BAAI/bge-base-en-v1.5"
EMBED_BATCH    = 64
RRF_K          = 60
XML_COLLECTION = "xml_corpus_invalid"


def get_device():
    if torch.cuda.is_available():
        print("[retriever] Using CUDA GPU")
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("[retriever] Using Apple MPS")
        return "mps"
    print("[retriever] Using CPU")
    return "cpu"


def tokenise(text):
    text = re.sub(r"[^a-zA-Z0-9_]", " ", (text or "").lower())
    return [t for t in text.split() if len(t) > 1]


def rrf(ranked_a, ranked_b, k=RRF_K):
    scores = {}
    for rank, doc_id in enumerate(ranked_a, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    for rank, doc_id in enumerate(ranked_b, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.__getitem__, reverse=True)


class BGEEmbeddingFunction(EmbeddingFunction):
    def __init__(self):
        device = get_device()
        print(f"[retriever] Loading {EMBED_MODEL} ...")
        self._model = SentenceTransformer(EMBED_MODEL, device=device)
        print("[retriever] Model loaded")

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = self._model.encode(
            list(input),
            batch_size           = EMBED_BATCH,
            normalize_embeddings = True,
            show_progress_bar    = False,
        )
        return embeddings.tolist()


class Retriever:
    def __init__(self, xml_bugs, persist_dir=CHROMA_DIR):
        self.xml_bugs  = xml_bugs
        self.xml_by_id = {b["id"]: b for b in xml_bugs}

        embed_fn      = BGEEmbeddingFunction()
        chroma_client = chromadb.PersistentClient(path=persist_dir)

        self._xml_col = chroma_client.get_or_create_collection(
            XML_COLLECTION, embedding_function=embed_fn,
            metadata={"hnsw:space": "cosine"},
        )
        self._xml_bm25     = None
        self._xml_bm25_ids = []

    def build(self, force_reindex=False):
        if self._xml_col.count() > 0 and not force_reindex:
            print(f"[retriever] XML corpus: {self._xml_col.count():,} docs already indexed — skipping")
        else:
            print(f"[retriever] Embedding {len(self.xml_bugs):,} XML bugs ...")
            for i in tqdm(range(0, len(self.xml_bugs), EMBED_BATCH), desc="XML corpus"):
                batch = self.xml_bugs[i : i + EMBED_BATCH]
                self._xml_col.add(
                    ids       = [d["id"]   for d in batch],
                    documents = [d["text"] for d in batch],
                    metadatas = [{
                        "title"     : d["title"][:500],
                        "package"   : d.get("package", ""),
                        "resolution": d.get("resolution", ""),
                    } for d in batch],
                )
            print(f"[retriever] XML corpus: {self._xml_col.count():,} docs indexed")

        self._xml_bm25_ids = [b["id"] for b in self.xml_bugs]
        self._xml_bm25     = BM25Okapi([tokenise(b["text"]) for b in self.xml_bugs])
        print("[retriever] Index ready")

    def get_similar_xml_bugs(self, issue, k=3):
        query_text = issue["text"]
        fetch      = min(k * 4, self._xml_col.count())

        dense_res = self._xml_col.query(
            query_texts = [query_text],
            n_results   = fetch,
            include     = ["distances"],
        )
        dense_ids = dense_res["ids"][0]

        scores     = self._xml_bm25.get_scores(tokenise(query_text))
        sparse_ids = [
            self._xml_bm25_ids[i]
            for i in sorted(range(len(scores)), key=lambda x: -scores[x])
        ][:fetch]

        merged = rrf(dense_ids, sparse_ids)[:k]
        return [self.xml_by_id[i] for i in merged if i in self.xml_by_id]


retriever = Retriever(xml_bugs)
retriever.build(force_reindex=False)

[retriever] Using CPU
[retriever] Loading BAAI/bge-base-en-v1.5 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[retriever] Model loaded
[retriever] XML corpus: 3,529 docs already indexed — skipping
[retriever] Index ready


In [7]:
# ============================================================
# CELL 6.5 — Initialize clients + test API connection
# ============================================================

from openai import OpenAI
try:
    import anthropic
except ImportError:
    !pip install -q anthropic
    import anthropic

# ── Build opt-out headers for OpenAI ─────────────────────────
_openai_headers = {}
if OPENAI_ORG_ID:
    _openai_headers["OpenAI-Organization"] = OPENAI_ORG_ID
if OPENAI_PROJECT_ID:
    _openai_headers["OpenAI-Project"] = OPENAI_PROJECT_ID

# ── Clients ───────────────────────────────────────────────────
openai_client    = OpenAI(
    api_key         = OPENAI_API_KEY,
    default_headers = _openai_headers,
)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print(f"OpenAI client  : set")
print(f"Anthropic client: {'set' if anthropic_client else 'not set (no key)'}")

# ── Models that use Responses API (GPT-5 and newer) ──────────
_RESPONSES_API_MODELS = {
    "gpt-5", "gpt-5-mini", "gpt-5.1", "gpt-5.4", "gpt-5.4-mini",
    "gpt-5.4-nano", "gpt-5.4-pro", "gpt-5.5", "gpt-5.5-pro",
    "o3", "o3-mini", "o4-mini",
}

# ── Test OpenAI connection ────────────────────────────────────
try:
    resp = openai_client.chat.completions.create(
        model      = "gpt-4o-mini",
        messages   = [{"role": "user", "content": "reply with the word ok"}],
        max_tokens = 5,
    )
    print(f"\nOpenAI API: working — {resp.choices[0].message.content}")
except Exception as e:
    print(f"\nOpenAI API error: {type(e).__name__}: {e}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 7.5 MB/s eta 0:00:00
OpenAI client  : set
Anthropic client: set

OpenAI API: working — Ok.


In [8]:
# ============================================================
# CELL IV1 — Invalid detection setup
# ============================================================

import pandas as pd
import re
import random
from collections import defaultdict

random.seed(42)

# ── Preprocessing: mask 'invalid' state to prevent label leakage ──
def mask_state(state):
    """Remove 'invalid' label from state to prevent leakage into prompt.
    true_invalid is preserved separately for evaluation only."""
    if str(state).strip().lower() == 'invalid':
        return ''
    return state

# ── Load genuine GitHub issues (75) ──────────────────────────
df = pd.read_csv(GITHUB_CSV)
df_github = df[~df['Title'].str.contains(
    r'bugzilla bug \d+', case=False, na=False
)].copy()
df_github['comments'] = df_github['Comments'].fillna(0).astype(int)
df_github['state']    = df_github['state'].fillna('')

print(f"Genuine GitHub issues : {len(df_github)}")
print(f"\nState distribution (original):")
print(df_github['state'].value_counts())
print(f"Null states           : {(df_github['state'] == '').sum()}")
print(f"\nKnown invalid         : {(df_github['state'] == 'invalid').sum()}")

# Build test issues list
invalid_test_issues = []
for _, row in df_github.iterrows():
    comments     = int(float(str(row.get('Comments', 0) or 0)))
    state_orig   = str(row.get('state') or '')
    state_masked = mask_state(state_orig)   # ← 'invalid' replaced with ''

    invalid_test_issues.append({
        "id"          : str(int(row["Issue Number"])),
        "title"       : str(row["Title"]),
        "package"     : str(row.get("package") or ""),
        "type"        : str(row.get("type") or ""),
        "comments"    : comments,
        "state"       : state_masked,         # ← masked — shown to model
        "true_invalid": state_orig == 'invalid',  # ← original — evaluation only
        "text"        : _clean(
            f"Title: {row['Title']}\n"
            f"Package: {row.get('package','')}\n"
            f"Comments: {comments}\n"
            f"State: {state_masked}"           # ← masked — shown to model
        ),
    })

print(f"\nTest issues ready     : {len(invalid_test_issues)}")
print(f"GT (true_invalid=True): {sum(1 for i in invalid_test_issues if i['true_invalid'])}")
print(f"State='invalid' in prompt: {sum(1 for i in invalid_test_issues if i['state'] == 'invalid')}  (should be 0)")

# ── Build example pools from XML corpus ──────────────────────
xml_invalid_examples = [
    b for b in xml_bugs
    if b.get('resolution', '').upper() == 'INVALID'
]
xml_valid_examples = [
    b for b in xml_bugs
    if b.get('resolution', '').upper() == 'FIXED'
]

print(f"\nXML INVALID examples : {len(xml_invalid_examples)}")
print(f"XML FIXED examples   : {len(xml_valid_examples)}")

print(f"\nSample INVALID XML bugs:")
for b in random.sample(xml_invalid_examples, min(5, len(xml_invalid_examples))):
    print(f"  #{b['id']}: {b['title'][:80]}")

Genuine GitHub issues : 75

State distribution (original):
state
needs-triage                                          16
needs-maintainer-feedback|needs-owner|needs-triage    14
needs-maintainer-feedback|needs-triage                13
needs-maintainer-feedback                             13
                                                      11
needs-owner|needs-triage                               6
needs-maintainer-feedback|needs-owner                  1
invalid                                                1
Name: count, dtype: int64
Null states           : 11

Known invalid         : 1

Test issues ready     : 75
GT (true_invalid=True): 1
State='invalid' in prompt: 0  (should be 0)

XML INVALID examples : 182
XML FIXED examples   : 2962

Sample INVALID XML bugs:
  #4312: MdeModulePkg/Bus/Scsi/ScsiDiskDxe: Coverity scan flag UNUSED_VALUE issues
  #167: EfiShellParametersProtocol always return 0 for Argc
  #942: function always return the same value
  #2694: Xhci Assert when Rese

In [11]:
import openai
from google.colab import userdata

client = openai.OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "hi"}],
    max_tokens=5
)
print("✅ Working:", response.choices[0].message.content)

✅ Working: Hello! How can I


In [13]:
# ============================================================
# CELL IV2 — Invalid detectors: all models, System A + B
# Saves JSON + CSV per model to RESULTS_DIR
# ============================================================

import json, time, os, csv, re
from tqdm.notebook import tqdm

random.seed(42)

# ── Few-shot examples ─────────────────────────────────────────
fewshot_invalid = random.sample(xml_invalid_examples, min(4, len(xml_invalid_examples)))
fewshot_valid   = random.sample(xml_valid_examples,   min(4, len(xml_valid_examples)))


def build_invalid_fewshot():
    block = "EXAMPLES FROM THIS PROJECT'S HISTORY:\n"
    for b in fewshot_invalid:
        block += (
            f"\nTitle      : {b['title']}\n"
            f"Package    : {b.get('package', '-')}\n"
            f"Resolution : INVALID\n"
        )
    for b in fewshot_valid:
        block += (
            f"\nTitle      : {b['title']}\n"
            f"Package    : {b.get('package', '-')}\n"
            f"Resolution : VALID\n"
        )
    return block


_INVALID_PROMPT_A = """You are an experienced EDK II firmware project maintainer.

Decide whether a GitHub issue is VALID or INVALID for the EDK II project tracker.

An issue is INVALID if:
- It is spam or completely unrelated to firmware development
  (e.g. printer support, social media, password reset, unrelated software)
- It is a general usage question, not a bug report or feature request
- It describes something that works as designed — the reporter misunderstood the behavior
- It does not contain enough information to reproduce or investigate

An issue is VALID if:
- It is a genuine bug report with a reproducible problem
- It is a legitimate feature request for EDK II
- It describes a real security vulnerability or build failure
- It has enough information to investigate
- It relates to EDK II build infrastructure such as build tools, compilers, or linkers
- It relates to EDK II coding style enforcement tools
- It involves any EDK II package, driver, protocol, or firmware component
- When in doubt, treat the issue as VALID

{fewshot}

Now classify the following issue.
Respond with ONLY valid JSON:
{{
  "is_invalid": <bool>,
  "confidence": <float 0.0-1.0>,
  "reason"    : <one sentence explaining why>
}}"""

_INVALID_PROMPT_B = """You are an experienced EDK II firmware project maintainer.

Decide whether a GitHub issue is VALID or INVALID for the EDK II project tracker.

An issue is INVALID if:
- It is spam or completely unrelated to firmware development
  (e.g. printer support, social media, password reset, unrelated software)
- It is a general usage question, not a bug report or feature request
- It describes something that works as designed


An issue is VALID if:
- It is a genuine bug report with a reproducible problem
- It is a legitimate feature request for EDK II
- It describes a real security vulnerability or build failure
- It has enough information to investigate
- It relates to EDK II build infrastructure such as build tools, compilers, or linkers
- It relates to EDK II coding style enforcement tools
- It involves any EDK II package, driver, protocol, or firmware component
- When in doubt, treat the issue as VALID

{fewshot}

The following SIMILAR HISTORICAL BUGS from this codebase show how issues
were classified as INVALID or VALID — use them to calibrate your decision:
{xml_context}

Now classify the following issue.
Respond with ONLY valid JSON:
{{
  "is_invalid": <bool>,
  "confidence": <float 0.0-1.0>,
  "reason"    : <one sentence explaining why>
}}"""


def fmt_issue_invalid(issue):
    comments_int = issue.get("comments", 0)
    try:
        comments_int = int(float(str(comments_int)))
        if comments_int == 0:
            comments_str = "0 (no discussion)"
        elif comments_int >= 8:
            comments_str = f"{comments_int} (high engagement)"
        elif comments_int >= 4:
            comments_str = f"{comments_int} (moderate engagement)"
        else:
            comments_str = f"{comments_int} (low engagement)"
    except:
        comments_str = str(comments_int)
    return (
        f"Title    : {issue['title']}\n"
        f"Package  : {issue.get('package', '-')}\n"
        f"Comments : {comments_str}\n"
        f"State    : {issue.get('state') or 'not set'}"
    )


def fmt_xml_invalid(bug):
    res   = bug.get("resolution", "").upper()
    label = "INVALID" if res == "INVALID" else "VALID"
    return f"  - [{label:8s}] {bug['title'][:80]}"


def parse_invalid_json(raw):
    raw = re.sub(r"```(?:json)?", "", raw).strip()
    raw = re.sub(r'\\(?!["\\\/bfnrt]|u[0-9a-fA-F]{4})', r'\\\\\\\\', raw)
    try:
        return json.loads(raw)
    except:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            text = m.group()
            text = re.sub(r'\\(?!["\\\/bfnrt]|u[0-9a-fA-F]{4})', r'\\\\\\\\', text)
            return json.loads(text)
        raise ValueError(f"Could not parse: {raw}")


def call_invalid_llm(prompt, user, model):
    """Unified LLM call — routes to OpenAI or Anthropic"""
    content  = prompt + "\n\n" + user
    provider = "anthropic" if model.startswith("claude") else "openai"
    use_responses_api = any(
        model.startswith(m) for m in _RESPONSES_API_MODELS
    )

    for attempt in range(3):
        try:
            if provider == "anthropic":
                # Anthropic API — never used for training per commercial terms
                resp = anthropic_client.messages.create(
                    model      = model,
                    max_tokens = 200,
                    messages   = [{"role": "user", "content": content}],
                )
                raw = resp.content[0].text
            elif use_responses_api:
                # OpenAI Responses API (GPT-5 and newer)
                resp = openai_client.responses.create(
                    model         = model,
                    input         = content,
                    extra_headers = _openai_headers,
                )
                raw = resp.output_text
            else:
                # OpenAI Chat Completions API (GPT-4 and older)
                resp = openai_client.chat.completions.create(
                    model         = model,
                    messages      = [{"role": "user", "content": content}],
                    temperature   = 0.0,
                    max_tokens    = 200,
                    extra_headers = _openai_headers,
                )
                raw = resp.choices[0].message.content

            if not raw or not raw.strip():
                raise ValueError("Empty response")

            result = parse_invalid_json(raw)
            result["is_invalid"] = bool(result.get("is_invalid", False))
            result["confidence"] = float(result.get("confidence", 0.0))
            return result

        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {type(e).__name__}: {e}")
            if attempt == 2:
                return {"is_invalid": False, "confidence": 0.0, "reason": "error"}
            time.sleep(2)


def predict_invalid_A(issue, model):
    fewshot = build_invalid_fewshot()
    prompt  = _INVALID_PROMPT_A.format(fewshot=fewshot)
    user    = "ISSUE TO CLASSIFY:\n" + fmt_issue_invalid(issue)
    return call_invalid_llm(prompt, user, model)


def predict_invalid_B(issue, model):
    similar   = retriever.get_similar_xml_bugs(issue, k=XML_CONTEXT_K)
    xml_block = "\n".join(fmt_xml_invalid(b) for b in similar)
    fewshot   = build_invalid_fewshot()
    prompt    = _INVALID_PROMPT_B.format(fewshot=fewshot, xml_context=xml_block)
    user      = "ISSUE TO CLASSIFY:\n" + fmt_issue_invalid(issue)
    return call_invalid_llm(prompt, user, model)


# ── Run all models — System A and B ──────────────────────────
summary = []

for model in ALL_MODELS:
    model_slug = model.replace("/", "-").replace(".", "_")

    for system, predict_fn, system_label in [
        ("A", predict_invalid_A, "no RAG"),
        ("B", predict_invalid_B, "with RAG"),
    ]:
        path_json = f'{RESULTS_DIR}/invalid_{system}_{model_slug}.json'
        path_csv  = f'{RESULTS_DIR}/invalid_{system}_{model_slug}.csv'

        print(f"\n{'='*55}")
        print(f"  System {system} ({system_label}) — {model}")
        print(f"{'='*55}")

        # Delete stale result files — always re-run
        for stale in [path_json, path_csv]:
            if os.path.exists(stale):
                os.remove(stale)

        results = []
        for issue in tqdm(invalid_test_issues, desc=f"{system} | {model}"):
            try:
                pred = predict_fn(issue, model)
                pred.update({
                    "model"       : model,
                    "issue_id"    : issue["id"],
                    "title"       : issue["title"],
                    "package"     : issue["package"],
                    "comments"    : issue["comments"],
                    "state"       : issue["state"],
                    "true_invalid": issue["true_invalid"],
                })
                results.append(pred)
            except Exception as e:
                print(f"  Skipped #{issue['id']}: {e}")

        # Save CSV
        if results:
            keys = ["model", "issue_id", "title", "package", "is_invalid",
                    "confidence", "reason", "true_invalid", "state"]
            with open(path_csv, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
                writer.writeheader()
                writer.writerows(results)

        n_inv = sum(1 for r in results if r["is_invalid"])
        print(f"  Flagged  : {n_inv}/{len(results)}")
        print(f"  Saved CSV: {path_csv}")
        summary.append({"model": model, "system": system,
                         "flagged": n_inv, "total": len(results)})

# ── Summary table ─────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  Summary — Invalid Detection")
print(f"{'='*65}")
print(f"  {'Model':35s} {'Sys':5s} {'Flagged':10s} {'Total':10s}")
print(f"  {'-'*63}")
for s in summary:
    print(f"  {s['model']:35s} {s['system']:5s} {s['flagged']:10d} {s['total']:10d}")
print(f"{'='*65}")
print(f"\nGround truth: 1 known invalid issue in 75 test issues")



  System A (no RAG) — gpt-4o-mini


A | gpt-4o-mini:   0%|          | 0/75 [00:00<?, ?it/s]

  Flagged  : 2/75
  Saved CSV: /content/drive/MyDrive/invalid_detect/results/invalid_A_gpt-4o-mini.csv

  System B (with RAG) — gpt-4o-mini


B | gpt-4o-mini:   0%|          | 0/75 [00:00<?, ?it/s]

  Flagged  : 1/75
  Saved CSV: /content/drive/MyDrive/invalid_detect/results/invalid_B_gpt-4o-mini.csv

  System A (no RAG) — gpt-4o


...

  Flagged  : 2/75
  Saved CSV: /content/drive/MyDrive/invalid_detect/results/invalid_B_claude-sonnet-4-6.csv

  System A (no RAG) — claude-opus-4-7


A | claude-opus-4-7:   0%|          | 0/75 [00:00<?, ?it/s]

  Flagged  : 1/75
  Saved CSV: /content/drive/MyDrive/invalid_detect/results/invalid_A_claude-opus-4-7.csv

  System B (with RAG) — claude-opus-4-7


B | claude-opus-4-7:   0%|          | 0/75 [00:00<?, ?it/s]

  Flagged  : 1/75
  Saved CSV: /content/drive/MyDrive/invalid_detect/results/invalid_B_claude-opus-4-7.csv

  Summary — Invalid Detection
  Model                               Sys   Flagged    Total     
  ---------------------------------------------------------------
  gpt-4o-mini                         A              2         75
  gpt-4o-mini                         B              1         75
  gpt-4o                              A              1         75
  gpt-4o                              B              1         75
  gpt-4.1-mini                        A              1         75
  gpt-4.1-mini                        B              1         75
  gpt-4.1                             A              1         75
  gpt-4.1                             B              1         75
  gpt-5.4-mini                        A              2         75
  gpt-5.4-mini                        B              1         75
  gpt-5.4                             A              2         75
  gp

In [14]:
# ============================================================
# CELL IV3 — Evaluation: Precision, Recall, F1, Accuracy
# Setup 1 only: GT = issues where state == 'invalid' in CSV
# N = 75, TN = 75 - TP - FP - FN
# F1_mac = (F1+ + F1-) / 2  (standard sklearn macro)
# Saves: invalid_evaluation_combined.csv  (predictions + metrics)
#        invalid_evaluation_summary.csv   (one row per model/system)
# ============================================================

import os, glob, csv
import pandas as pd

N = 75

# ── Ground truth: issues labeled state='invalid' in the CSV ──
GT1 = {
    str(i["id"])
    for i in invalid_test_issues
    if i["true_invalid"]
}
print(f"Ground truth (Setup 1): {GT1}  — {len(GT1)} confirmed invalid issue(s)")


def compute_metrics(flagged_ids, gt_set, n=N):
    flagged = set(str(i) for i in flagged_ids)
    gt      = set(str(i) for i in gt_set)

    tp = len(flagged &  gt)
    fp = len(flagged - gt)
    fn = len(gt - flagged)
    tn = n - tp - fp - fn

    p_pos  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r_pos  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    p_neg  = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    r_neg  = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_pos = 2 * p_pos * r_pos / (p_pos + r_pos) if (p_pos + r_pos) > 0 else 0.0
    f1_neg = 2 * p_neg * r_neg / (p_neg + r_neg) if (p_neg + r_neg) > 0 else 0.0
    p_mac  = (p_pos + p_neg) / 2
    r_mac  = (r_pos + r_neg) / 2
    f1_mac = (f1_pos + f1_neg) / 2
    acc    = (tp + tn) / n

    return dict(
        tp=tp, fp=fp, fn=fn, tn=tn,
        p_pos=round(p_pos,4), r_pos=round(r_pos,4),
        p_neg=round(p_neg,4), r_neg=round(r_neg,4),
        p_mac=round(p_mac,4), r_mac=round(r_mac,4),
        f1_pos=round(f1_pos,4), f1_neg=round(f1_neg,4),
        f1_mac=round(f1_mac,4), acc=round(acc,4),
    )


# ── Load all prediction CSVs ──────────────────────────────────
csv_files = sorted(glob.glob(f"{RESULTS_DIR}/invalid_*.csv"))
if not csv_files:
    print(f"No prediction CSVs found in {RESULTS_DIR}. Run CELL IV2 first.")
else:
    print(f"Found {len(csv_files)} prediction files.\n")

combined_rows = []
summary_rows  = []

for path in csv_files:
    fname      = os.path.basename(path)
    parts      = fname.replace(".csv", "").split("_", 2)
    system     = parts[1] if len(parts) > 1 else "?"
    model_slug = parts[2] if len(parts) > 2 else fname
    model_name = model_slug.replace("_", ".")

    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"  ERROR reading {fname}: {e}")
        continue

    # All flagged issues (content-based — no confidence threshold)
    flagged_ids = set(
        str(r["issue_id"])
        for _, r in df.iterrows()
        if r["is_invalid"] == True
    )

    m = compute_metrics(flagged_ids, GT1)

    # ── Per-issue rows (prediction + metrics) ─────────────────
    for _, row in df.iterrows():
        issue_id  = str(row["issue_id"])
        in_gt     = issue_id in GT1
        predicted = bool(row["is_invalid"])
        combined_rows.append({
            "model"          : model_name,
            "system"         : system,
            "issue_id"       : issue_id,
            "title"          : row.get("title", ""),
            "package"        : row.get("package", ""),
            "is_invalid_pred": predicted,
            "confidence"     : row.get("confidence", ""),
            "reason"         : row.get("reason", ""),
            "true_invalid"   : in_gt,
            "correct"        : (predicted == in_gt),
            # Model-level metrics repeated per row for easy filtering
            "tp"     : m["tp"],    "fp"   : m["fp"],
            "fn"     : m["fn"],    "tn"   : m["tn"],
            "p_pos"  : m["p_pos"], "r_pos": m["r_pos"],
            "p_neg"  : m["p_neg"], "r_neg": m["r_neg"],
            "p_mac"  : m["p_mac"], "r_mac": m["r_mac"],
            "f1_pos" : m["f1_pos"],"f1_neg": m["f1_neg"],
            "f1_mac" : m["f1_mac"],"acc"  : m["acc"],
        })

    summary_rows.append({
        "model": model_name, "system": system,
        "flagged": len(flagged_ids), **m
    })

    print(
        f"  {model_name:<16} Sys {system} | "
        f"Flagged={len(flagged_ids):>2} "
        f"TP={m['tp']} FP={m['fp']} FN={m['fn']} TN={m['tn']} | "
        f"P+={m['p_pos']:.3f} R+={m['r_pos']:.3f} "
        f"F1_mac={m['f1_mac']:.3f} Acc={m['acc']:.3f}"
    )

# ── Save combined CSV ─────────────────────────────────────────
combined_path = f"{RESULTS_DIR}/invalid_evaluation_combined.csv"
if combined_rows:
    fields = [
        "model", "system", "issue_id", "title", "package",
        "is_invalid_pred", "confidence", "reason",
        "true_invalid", "correct",
        "tp", "fp", "fn", "tn",
        "p_pos", "r_pos", "p_neg", "r_neg",
        "p_mac", "r_mac", "f1_pos", "f1_neg", "f1_mac", "acc",
    ]
    with open(combined_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(combined_rows)
    print(f"\n✓ Predictions + metrics → {combined_path}")

# ── Save summary CSV ──────────────────────────────────────────
summary_path = f"{RESULTS_DIR}/invalid_evaluation_summary.csv"
if summary_rows:
    fields_sum = [
        "model", "system", "flagged",
        "tp", "fp", "fn", "tn",
        "p_pos", "r_pos", "p_neg", "r_neg",
        "p_mac", "r_mac", "f1_pos", "f1_neg", "f1_mac", "acc",
    ]
    with open(summary_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields_sum)
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"✓ Summary table          → {summary_path}")

# ── Print summary table ───────────────────────────────────────
print(f"\n{'='*88}")
print(f"  INVALID DETECTION SUMMARY  |  GT={{state=invalid}}  |  N=75")
print(f"  F1_mac = (F1+ + F1-) / 2  |  TN = 75 - TP - FP - FN")
print(f"{'='*88}")
print(f"  {'Model':<18} {'Sys':<4} {'Flag':>4} {'TP':>3} {'FP':>3} {'FN':>3} {'TN':>4} "
      f"{'P+':<7} {'R+':<7} {'P_mac':<7} {'R_mac':<7} {'F1_mac':<8} {'Acc'}")
print(f"  {'-'*84}")
for r in sorted(summary_rows, key=lambda x: (x['system'], x['model'])):
    print(
        f"  {r['model']:<18} {r['system']:<4} {r['flagged']:>4} "
        f"{r['tp']:>3} {r['fp']:>3} {r['fn']:>3} {r['tn']:>4} "
        f"{r['p_pos']:<7.3f} {r['r_pos']:<7.3f} "
        f"{r['p_mac']:<7.3f} {r['r_mac']:<7.3f} "
        f"{r['f1_mac']:<8.3f} {r['acc']:.3f}"
    )
print(f"\nNote: P-, R-, Acc dominated by TN (~73-74/75). Primary metrics: P+, R+, F1_mac")
print(f"Threat to validity: 74 unreviewed issues assumed valid → TN may be inflated.")

Ground truth (Setup 1): {'10579'}  — 1 confirmed invalid issue(s)
Found 20 prediction files.

  claude-haiku-4-5 Sys A | Flagged= 1 TP=0 FP=1 FN=1 TN=73 | P+=0.000 R+=0.000 F1_mac=0.493 Acc=0.973
  claude-opus-4-7  Sys A | Flagged= 1 TP=0 FP=1 FN=1 TN=73 | P+=0.000 R+=0.000 F1_mac=0.493 Acc=0.973
  claude-sonnet-4-6 Sys A | Flagged= 2 TP=0 FP=2 FN=1 TN=72 | P+=0.000 R+=0.000 F1_mac=0.490 Acc=0.960
  gpt-4.1-mini     Sys A | Flagged= 1 TP=0 FP=1 FN=1 TN=73 | P+=0.000 R+=0.000 F1_mac=0.493 Acc=0.973
  gpt-4.1          Sys A | Flagged= 1 TP=0 FP=1 FN=1 TN=73 | P+=0.000 R+=0.000 F1_mac=0.493 Acc=0.973
  gpt-4o-mini      Sys A | Flagged= 2 TP=0 FP=2 FN=1 TN=72 | P+=0.000 R+=0.000 F1_mac=0.490 Acc=0.960
  gpt-4o           Sys A | Flagged= 1 TP=0 FP=1 FN=1 TN=73 | P+=0.000 R+=0.000 F1_mac=0.493 Acc=0.973
  gpt-5.4-mini     Sys A | Flagged= 2 TP=0 FP=2 FN=1 TN=72 | P+=0.000 R+=0.000 F1_mac=0.490 Acc=0.960
  gpt-5.4          Sys A | Flagged= 2 TP=0 FP=2 FN=1 TN=72 | P+=0.000 R+=0.000 F1_mac=0.4

Each issue gets its own unique set of 3 bugs tailored to its specific topic. The retrieval is dynamic — ChromaDB finds the 3 closest matches in the XML corpus for that particular issue's text.
This is why RAG is powerful — it's not one fixed context for all issues. It's personalized context per issue. A bug about networking gets networking-related historical examples. A bug about SMM gets SMM-related historical examples. The LLM always has the most relevant historical evidence for whatever it's currently classifying.